# Summary

Test calling a deployed agent

In [2]:
import os, sys
import json
import time

from vertexai import agent_engines

import vertexai

# Import libraries from the Agent Development Kit
from google.adk.agents import Agent
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.sessions import VertexAiSessionService
from google.genai import types


utils_path = "../interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication

# Set environment variables
dotenv_path = "../data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()

# Initialize Vertex AI API once per session
vertexai.init(project=os.environ["GOOGLE_CLOUD_PROJECT"],
              location=os.environ["GOOGLE_CLOUD_LOCATION"],
              staging_bucket=os.environ["STAGING_BUCKET"])



## Retrieve a deployed agent

In [3]:
dir(agent_engines)

# get a list of agents
for agent in agent_engines.list():
    print(agent.resource_name)

# Delete if needed
# for agent in agent_engines.list():
#     agent_engines.delete(resource_name=agent.resource_name,
#                          force=True)


projects/1062597788108/locations/us-central1/reasoningEngines/2143827771837644800
projects/1062597788108/locations/us-central1/reasoningEngines/4982010330755366912


In [4]:
os.getenv("AGENT_ENGINE_ID")

'projects/1062597788108/locations/us-central1/reasoningEngines/2143827771837644800'

In [4]:
# dir(agent_engines)
# dir(agent_engines.AgentEngine)
# # agent_engines.AgentEngine
#
# # agent_engines.AgentEngine.list()
# agent_engines.AgentEngine.create()


In [5]:

# Retreive an ADK agent
# agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))
agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))


# dir(agent_engine)
# Opt 1: Create a session using agent_engine
session = agent_engine.create_session(user_id="u_123")



#### Option 2
# session_service = VertexAiSessionService(project=os.environ["GOOGLE_CLOUD_PROJECT"],
#                                          location=os.environ["GOOGLE_CLOUD_LOCATION"])
#
# # type(agent_engine)
# session_service_active1 = agent_engine.create_session(user_id="u_123")


In [5]:
type(agent_engine)
dir(agent_engine)
agent_engine.operation_schemas()


[{'name': 'get_session',
  'description': 'Get a session for the given user.',
  'api_mode': '',
  'parameters': {'type': 'object',
   'properties': {'user_id': {'type': 'string'},
    'session_id': {'type': 'string'}},
   'required': ['user_id', 'session_id']}},
 {'name': 'list_sessions',
  'description': 'List sessions for the given user.',
  'api_mode': '',
  'parameters': {'type': 'object',
   'properties': {'user_id': {'type': 'string'}},
   'required': ['user_id']}},
 {'name': 'create_session',
  'description': 'Creates a new session.',
  'api_mode': '',
  'parameters': {'type': 'object',
   'properties': {'user_id': {'type': 'string'},
    'state': {'type': 'object', 'nullable': True},
    'session_id': {'type': 'string', 'nullable': True}},
   'required': ['user_id']}},
 {'name': 'delete_session',
  'description': 'Deletes a session for the given user.',
  'api_mode': '',
  'parameters': {'type': 'object',
   'properties': {'user_id': {'type': 'string'},
    'session_id': {'typ

In [6]:
agent_engine

resource name: projects/1062597788108/locations/us-central1/reasoningEngines/136700081658134528

In [9]:
# # from google.adk.sessions import VertexAiSessionService
# #
# app_name=os.getenv("AGENT_ENGINE_ID")
# user_id="u_123"
#
# # Create the ADK runner with VertexAiSessionService
# session_service = VertexAiSessionService(project=os.environ["GOOGLE_CLOUD_PROJECT"],
#                                          location=os.environ["GOOGLE_CLOUD_LOCATION"])
# runner = Runner(
#     # agent=root_agent,
#     agent=agent_engine,
#     app_name=app_name,
#     session_service=session_service)
#
#
# # Helper method to send query to the runner
# def call_agent(query, session_id, user_id):
#   content = types.Content(role='user', parts=[types.Part(text=query)])
#   events = runner.run(
#       user_id=user_id, session_id=session_id, new_message=content)
#
#   for event in events:
#       if event.is_final_response():
#           final_response = event.content.parts[0].text
#           print("Agent Response: ", final_response)
#
# q1 = "What are LEAP exams?"
# # q2 = "What are the responsibilities of the board members of a California community college?"
#
# test_result2 = call_agent(query=q1, session_id=user_id="U_123")

In [4]:
# type(agent_engine)
# dir(session_service)

# session.keys()

## Test the agent with one query

In [6]:
q1 = "What are LEAP exams?"
q1 = "What are the responsibilities of the board members of a California community college?"

test_result = agent_engine.stream_query(message=q1, user_id="U_123")


# test_result = agent_engine.async_stream_query(message=q1, user_id="U_123")

events = []
for event in test_result:
    events.append(event)



In [7]:
for event in events:
    print(type(event))
    print(event.keys())
    print(event['content'].keys())
    # print(event['content']['parts'])
    # print(event['grounding_metadata'])
    print(event['author'])
    print(event['actions'])
    print()




<class 'dict'>
dict_keys(['content', 'grounding_metadata', 'usage_metadata', 'invocation_id', 'author', 'actions', 'id', 'timestamp'])
dict_keys(['parts', 'role'])
rag_search_agent
{'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}}

<class 'dict'>
dict_keys(['content', 'grounding_metadata', 'usage_metadata', 'invocation_id', 'author', 'actions', 'id', 'timestamp'])
dict_keys(['parts', 'role'])
search_assistant
{'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}}

<class 'dict'>
dict_keys(['content', 'usage_metadata', 'invocation_id', 'author', 'actions', 'id', 'timestamp'])
dict_keys(['parts', 'role'])
synthesis_agent
{'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}}



In [8]:
for event in events:
    if event["author"] == "rag_search_agent":
        print(event["author"])
        e1 = event
        for entry in event["content"]["parts"]:
            try:
                print(entry["text"])
            except:
                pass

rag_search_agent
California's community college board of trustees are responsible for governing individual community college districts, adapting their district's mission to community needs, and ensuring college excellence and sustainability [5]. The League Board of Directors manages the organization's business affairs and establishes and evaluates the annual education policy agenda and legislative program [2]. The California Community College Trustees (CCCT) Board formulates education policy issues that come before the California Community Colleges Board of Governors, the State Legislature, and other relevant state-level boards and commissions [2].

Citations:
1) Trustees - Community College League of California
2) gs://ccc-crawl_data_xb/crawl_data/jsonl_files/wwwccleagueorg_2025May01_text.jsonl


In [83]:
e1

{'content': {'parts': [{'text': "California's community college board of trustees are responsible for governing individual community college districts, adapting their district's mission to community needs, and ensuring college excellence and sustainability [5]. The League Board of Directors manages the organization's business affairs and establishes and evaluates the annual education policy agenda and legislative program [2]. The California Community College Trustees (CCCT) Board formulates education policy issues that come before the California Community Colleges Board of Governors, the State Legislature, and other relevant state-level boards and commissions [2].\n\nCitations:\n1) Trustees - Community College League of California\n2) gs://ccc-crawl_data_xb/crawl_data/jsonl_files/wwwccleagueorg_2025May01_text.jsonl"}],
  'role': 'model'},
 'grounding_metadata': {'grounding_chunks': [{'retrieved_context': {'rag_chunk': {'text': 'id ccleague_79\nstructData title Trustees - Community Coll

In [38]:
e1.keys()

for key in e1.keys():
    # print(key)
    # print(e1[key])
    # print()
    if key == "grounding_metadata":
        print(e1[key].keys())
        print()
        for gc in e1[key]:
            # print(gc)
            # print(e1[key][gc])
            # print()
            #
            # get urls

            # get metadata
            if gc == "grounding_chunks":
                for gcli in e1[key]["grounding_chunks"]:
                    # print(gcli["retrieved_context"])

                    for gclikey in gcli["retrieved_context"].keys():

                        for gcl4key in gcli["retrieved_context"][gclikey].keys():
                            print(gcl4key)
                            print(type(gcli["retrieved_context"][gclikey][gcl4key]))
                            print("====")

                        # print(gclikey)
                        # print(type(gcli["retrieved_context"][gclikey]))
                        # print(gcli["retrieved_context"][gclikey])
                        #
                        # print("====")

    #


dict_keys(['grounding_chunks', 'grounding_supports', 'retrieval_queries'])

text
<class 'str'>
====


AttributeError: 'str' object has no attribute 'keys'

In [75]:
import re

finds_list = []
for item in e1["grounding_metadata"]["grounding_chunks"]:
    # print(type(item["retrieved_context"]["rag_chunk"]["text"]))
    # print(item["retrieved_context"]["rag_chunk"]["text"])
    # print("===")

    pats = {"title": r"structData title (.*)", "page_url": r"structData page_url (.*)"}


    for patkey in pats:

        patfinds = re.findall(pats[patkey], item["retrieved_context"]["rag_chunk"]["text"])
        if patfinds:
            for patfind in patfinds:
                finds_list.append({patkey: patfind})



In [82]:
e1["grounding_metadata"]["grounding_chunks"][3]

{'retrieved_context': {'rag_chunk': {'text': 'structData text The League Board of Directors is responsible for the management of the business affairs of the organization and also establishes and evaluates the annual education policy agenda and annual legislative program. Board Information All new agendas and minutes can now be viewed at: http:  www.boarddocs.com ca cclca bod Board.nsf Public If you have any questions please contact Agnes Lupa at agnes@ccleague.org. Julianna Barnes Immediate Past Chair, South Orange County CCD (CEOCCC) Nan Gomez-Heitzeberg Board Member, Kern CCD (CCCT) Deborah Knowles 2nd Vice Chair Secretary, Sacramento City College (CCCCS) Roger Schultz Chair, Mt. San Jacinto CCD (CEOCCC) Suzanne Lee Chan Board Member, Ohlone CCD (CCCT) Sunita Cooke Board Member, Mira Costa College CCD (CEOCCC) Willy Duncan Board Member, Sierra College (CEOCCC) Barbara Dunsheath Board Member, Cerritos CCD (CCCT) Andra Hoffman Board Member, Los Angeles CCD (CCCT) Amparo Medina Board Me

In [76]:
finds_list

# t2 = finds[1]
# dir(t2)
#
# t2.string

# find_between(item["retrieved_context"]["rag_chunk"]["text"], "StructData", "StructData")

[{'title': 'Trustees - Community College League of California'},
 {'page_url': 'https://www.ccleague.org/trustees/'}]

In [64]:
from google.cloud import discoveryengine_v1alpha as discoveryengine
from google.api_core.client_options import ClientOptions
import functions_framework
import json

# Your Project, Location, and Data Store ID
PROJECT_ID = "your-gcp-project-id"
LOCATION = "global"  # Or your specific location
DATA_STORE_ID = "your-data-store-id"

@functions_framework.http
def search_with_metadata(request):
    """
    Receives a query from a Vertex Agent and returns a response
    with structured metadata.
    """
    request_json = request.get_json(silent=True)
    query = request_json['sessionInfo']['parameters']['query'] # Adjust based on how agent sends params

    if not query:
        return json.dumps({"fulfillment_response": {"messages": [{"text": {"text": ["Please provide a search query."]}}]}})

    client = discoveryengine.SearchServiceClient(
        client_options=ClientOptions(api_endpoint=f"{LOCATION}-discoveryengine.googleapis.com")
    )
    serving_config = client.serving_config_path(
        project=PROJECT_ID, location=LOCATION,
        data_store=DATA_STORE_ID, serving_config="default_config"
    )

    search_request = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=query,
        page_size=3 # Get top 3 results
    )

    response = client.search(search_request)

    # Format the response for the agent
    final_response = "Here are the top results for your query:\n\n"
    for i, result in enumerate(response.results):
        doc_data = result.document.struct_data
        final_response += f"--- Result {i+1} ---\n"
        final_response += f"Snippet: ...{result.document.derived_struct_data['snippets'][0]['snippet']}...\n"
        final_response += f"Author: {doc_data.get('author', 'N/A')}\n"
        final_response += f"Source URL: {doc_data.get('source_url', 'N/A')}\n"
        final_response += f"Document ID: {result.document.id}\n\n"


    # Create the response JSON for Vertex AI Agent Builder
    response_json = {
        "fulfillment_response": {
            "messages": [
                {
                    "text": {
                        "text": [final_response]
                    }
                }
            ]
        }
    }

    return json.dumps(response_json)

['example1', 'example2']
['123banana456']


In [56]:
# agent_engines.get("4226237373903536128")
# dir(agent_engines)

# dir(agent_engine)
#
# test_session = agent_engine.get_session(user_id="U_123", session_id="4226237373903536128")
#
# dir(test_session)
#
# dir(test_session.items())



In [9]:
# for event in events:
#     if event["author"] == "synthesis_agent":
#         for entry in event["content"]["parts"]:
#             print(entry["text"])
#
for event in events:
#     # if event["author"] == "synthesis_agent":
    print(event["author"])
    for entry in event["content"]["parts"]:
        try:
            print(entry["text"])
        except:
            pass
#
    print()
    print()


# events


rag_search_agent
California's community college board of trustees are responsible for governing individual community college districts within the system. As locally elected officials, they adapt their district's mission to community needs and ensure college excellence and sustainability.

The League Board of Directors manages the organization's business affairs, establishes and evaluates the annual education policy agenda and legislative program. The California Community College Trustees (CCCT) Board formulates education policy issues that come before the California Community Colleges Board of Governors, the State Legislature, and other relevant state-level boards and commissions, and provides input to the League Board. The Chief Executive Officers of the California Community Colleges (CEOCCC) Board also takes positions on and formulates education policy issues that come before the California Community Colleges Board of Governors, the State Legislature, and other relevant state-level b

## Test the agent with multiple queries in a session

In [31]:
def pretty_print_event(event):
    """Pretty prints an event with truncation for long content."""
    if "content" not in event:
        print(f"[{event.get('author', 'unknown')}]: {event}")
        return

    author = event.get("author", "unknown")
    parts = event["content"].get("parts", [])

    for part in parts:
        if "text" in part:
            text = part["text"]
            # Truncate long text to 200 characters
            if len(text) > 200:
                text = text[:197] + "..."
            print(f"[{author}]: {text}")
        elif "functionCall" in part:
            func_call = part["functionCall"]
            print(f"[{author}]: Function call: {func_call.get('name', 'unknown')}")
            # Truncate args if too long
            args = json.dumps(func_call.get("args", {}))
            if len(args) > 100:
                args = args[:97] + "..."
            print(f"  Args: {args}")
        elif "functionResponse" in part:
            func_response = part["functionResponse"]
            print(f"[{author}]: Function response: {func_response.get('name', 'unknown')}")
            # Truncate response if too long
            response = json.dumps(func_response.get("response", {}))
            if len(response) > 100:
                response = response[:97] + "..."
            print(f"  Response: {response}")

agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))
print(f"Agent ID: Id='{agent_engine.resource_name}'")

# Create a session
user_id = "u_123"
session = agent_engine.create_session(user_id=user_id)
print(f"Session created: User='{user_id}', Session='{session['id']}'")

# Create some queries
queries = [
    "Hi, how are you?",
    "What are LEAP exams?",
    "What is the California Community Colleges Chancellor’s Office doing to ensure accessibility of its websites"
]

# Look for events for each query
for query in queries:
    print(f"\n[user]: {query}")
    for event in agent_engine.stream_query(
        user_id=user_id,
        session_id=session["id"],
        message=query,
    ):
        print("We got here by finding an event")
        pretty_print_event(event)
        print(session.get(session["id"]))


Agent ID: Id='projects/1062597788108/locations/us-central1/reasoningEngines/5338815048108212224'
Session created: User='u_123', Session='5108327174356598784'

[user]: Hi, how are you?
We got here by finding an event
[ask_rag_agent]: I am doing well, thank you for asking! How are you today?

None

[user]: What are LEAP exams?
We got here by finding an event
[ask_rag_agent]: LEAP exams are an alternate examination and appointment process for the recruitment and hiring of individuals with disabilities into State service. Please contact the Department of Rehabilitation f...
None

[user]: What is the California Community Colleges Chancellor’s Office doing to ensure accessibility of its websites
We got here by finding an event
[ask_rag_agent]: The California Community Colleges Chancellor’s Office is committed to ensuring digital accessibility for people with disabilities by continually improving the user experience and applying the relev...
None


## Look at the sessions

In [7]:
agent_engine.list_sessions(user_id=user_id)

{'sessions': [{'events': [],
   'user_id': 'u_123',
   'state': {},
   'id': '8957849324595707904',
   'app_name': '5338815048108212224',
   'last_update_time': 1746668523.502969},
  {'events': [],
   'user_id': 'u_123',
   'state': {},
   'id': '3558033371378483200',
   'app_name': '5338815048108212224',
   'last_update_time': 1746668232.507558}]}

## Create - This stuff doesn't work with the deployed agent but does work if the deployed agent is called directly from code

In [12]:

def send_query_to_agent(agent, query, user_id):
    """Sends a query to the specified agent and prints the response.

    Args:
        agent: The agent to send the query to.
        query: The query to send to the agent.

    Returns:
        A tuple containing the elapsed time (in milliseconds) and the final response from the agent.
    """

    # Create a new session - if you want to keep the history of interruction you need to move the
    # creation of the session outside of this function. Here we create a new session per query
    session = session_service.create_session(app_name=AGENT_APP_NAME,
                                             user_id=user_id,)
    # Create a content object representing the user's query
    print('\nUser Query: ', query)
    content = types.Content(role='user', parts=[types.Part(text=query)])

    # Start a timer to measure the response time
    start_time = time.time()

    # Create a runner object to manage the interaction with the agent
    runner = Runner(app_name=AGENT_APP_NAME, agent=agent, artifact_service=artifact_service, session_service=session_service)

    # Run the interaction with the agent and get a stream of events
    events = runner.run(user_id=user_id, session_id=session.id, new_message=content)

    final_response = None
    elapsed_time_ms = 0.0

    # Loop through the events returned by the runner
    for _, event in enumerate(events):

        is_final_response = event.is_final_response()

        if not event.content:
             continue

        if is_final_response:
            end_time = time.time()
            elapsed_time_ms = round((end_time - start_time) * 1000, 3)

            print("-----------------------------")
            print('>>> Inside final response <<<')
            print("-----------------------------")
            final_response = event.content.parts[0].text # Get the final response from the agent
            print(f'Agent: {event.author}')
            print(f'Response time: {elapsed_time_ms} ms\n')
            print(f'Final Response:\n{final_response}')
            print("----------------------------------------------------------\n")

    return elapsed_time_ms, final_response



In [4]:
MODEL='gemini-2.0-flash-001'

# # Create a basic agent with instructions amd greeting only
# basic_agent = Agent(model=MODEL,
#     name="agent_basic",
#     description="This agent responds to inquiries about its creation by stating it was built using the Google Agent Framework.",
#     instruction="If they ask you how you were created, tell them you were created with the Google Agent Framework.",
#     generate_content_config=types.GenerateContentConfig(temperature=0.2),
# )

############## Import the agent code from the python file used to create the deployed agent
rag_path = "ccc_chatbot/"
sys.path.insert(0, rag_path)
print(os.listdir(rag_path))
from agent import root_agent

basic_agent = root_agent

# Get the agent
# basic_agent = agent_engines.get(os.getenv("AGENT_ENGINE_ID2"))

# Create session and memory services
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# AGENT_APP_NAME = basic_agent.resource_name
AGENT_APP_NAME = 'agent_basic'
user_id = "u_123"

# Send a single query to the agent
send_query_to_agent(basic_agent, "How are you today", user_id)


['__init__.py', '__pycache__', 'agent.py', 'prompt.py', 'sub_agents']


ImportError: cannot import name 'root_agent' from partially initialized module 'agent' (most likely due to a circular import) (/Users/stephengodfrey/Documents/Workbench/Numantic/ccc-policy_assistant/test_agent_workflow/ccc_chatbot/agent.py)

In [32]:
from google.adk.agents import SequentialAgent
from google.adk.agents import LlmAgent

basic_agent = agent_engines.get(os.getenv("AGENT_ENGINE_ID2"))

code_pipeline_agent = SequentialAgent(
    name="run_other_agents",
    # sub_agents=[basic_agent],
    tools=[basic_agent],
    description="Wrapper for the basic agent",
)

# Create session and memory services
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# AGENT_APP_NAME = basic_agent.resource_name
AGENT_APP_NAME = 'agent_basic'
user_id = "u_123"

# Send a single query to the agent
send_query_to_agent(code_pipeline_agent, "How are you today", user_id)

ValidationError: 1 validation error for SequentialAgent
tools
  Extra inputs are not permitted [type=extra_forbidden, input_value=[<vertexai.agent_engines....nes/4194584083407306752], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/extra_forbidden